In [11]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# ===== SETUP: run this first, always =====
import os
from dotenv import load_dotenv
load_dotenv()

import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from groq import Groq

print("All imports loaded successfully")

All imports loaded successfully


In [13]:
from groq import Groq

# Create a "client" - this is like opening a connection to Groq's AI service
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Send a message to the AI model and get its response back
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # updated model name — smaller, fast, good for testing
    messages=[
        {"role": "user", "content": "Explain what RAG (Retrieval-Augmented Generation) is in one simple sentence."}
    ]
)

# Print just the AI's reply text
print(response.choices[0].message.content)

RAG is a technique that boosts AI text generation by first pulling relevant information from a database and then weaving that data into the answer.


In [14]:
%pip install pdfplumber


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Import pdfplumber - a library that reads PDF files and extracts their text
import pdfplumber

# Open the PDF file
with pdfplumber.open("data/Acme_Corp_Leave_Policy.pdf") as pdf:
    # Create an empty string to collect all the text
    full_text = ""
    
    # Loop through every page in the PDF
    for page in pdf.pages:
        # Extract the text from this page and add it to our full_text
        full_text += page.extract_text()
        full_text += "\n"  # add a line break between pages

# Print how much text we extracted, and a preview
print(f"Total characters extracted: {len(full_text)}")
print("\n--- First 500 characters ---\n")
print(full_text[:500])

Total characters extracted: 3821

--- First 500 characters ---

Acme Corporation
Employee Leave & Time-Off Policy — Effective January 2026
1. Purpose
This policy outlines the types of leave available to Acme Corporation employees, eligibility criteria,
accrual rules, and the process for requesting time off. It applies to all full-time, permanent
employees across all departments and locations.
2. Casual Leave
Employees are entitled to 12 (twelve) casual leave days per calendar year, credited at the rate of 1
day per month. Casual leave is intended for short-t


In [16]:
# %pip install langchain - only needed once
%pip install langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Import the text splitter tool - now from its own dedicated package
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a splitter:
# chunk_size = max characters per chunk
# chunk_overlap = how many characters repeat between chunks, so context isn't lost at the boundary
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Split our full_text into a list of smaller chunks
chunks = splitter.split_text(full_text)

# See how many chunks we got, and preview the first two
print(f"Number of chunks: {len(chunks)}")
print("\n--- Chunk 1 ---\n")
print(chunks[0])
print("\n--- Chunk 2 ---\n")
print(chunks[1])

Number of chunks: 10

--- Chunk 1 ---

Acme Corporation
Employee Leave & Time-Off Policy — Effective January 2026
1. Purpose
This policy outlines the types of leave available to Acme Corporation employees, eligibility criteria,
accrual rules, and the process for requesting time off. It applies to all full-time, permanent
employees across all departments and locations.
2. Casual Leave
Employees are entitled to 12 (twelve) casual leave days per calendar year, credited at the rate of 1

--- Chunk 2 ---

day per month. Casual leave is intended for short-term personal needs and cannot be availed for
more than 3 consecutive days without prior approval from the reporting manager.
Unused casual leave cannot be carried forward to the next calendar year and will lapse on
December 31st.
3. Sick Leave
Employees are entitled to 10 (ten) sick leave days per calendar year. A medical certificate is
required for sick leave exceeding 2 consecutive days. Sick leave may be carried forward up to a


In [19]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
# Import the sentence-transformers library — this handles turning text into vectors
from sentence_transformers import SentenceTransformer

# Load a small, fast, free embedding model
# "all-MiniLM-L6-v2" is a popular lightweight model - good balance of speed and accuracy
# This downloads the model the first time (small, ~80MB) and caches it locally after
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all our text chunks into embeddings (vectors of numbers)
chunk_embeddings = embedding_model.encode(chunks)

# Let's inspect what we got
print(f"Number of embeddings: {len(chunk_embeddings)}")
print(f"Each embedding has {len(chunk_embeddings[0])} numbers (dimensions)")
print("\n--- First 10 numbers of Chunk 1's embedding ---\n")
print(chunk_embeddings[0][:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of embeddings: 10
Each embedding has 384 numbers (dimensions)

--- First 10 numbers of Chunk 1's embedding ---

[-0.02267718 -0.02150293  0.03544436  0.04187777  0.0355714   0.06347182
  0.03760613 -0.00573383 -0.05914726 -0.01336355]


In [21]:
%pip install chromadb

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/23.5 MB 5.0 MB/s eta 0:00:05
   --- ------------------------------------ 1.8/23.5 MB 4.8 MB/s eta 0:00:05
   ---- ----------------------------------- 2.9/23.5 MB 4.5 MB/s eta 0:00:05
   ------ --------------------------------- 3.9/23.5 MB 4.6 MB/s eta 0:00:05
   -------- ------------------------------- 5.0/23.5 MB 4.7 MB/s eta 0:00:04
   ---------- ----------------------------- 6.0/23.5 MB 4.9 MB/s eta 0:00:04
   ------------ --------------------------- 7.1/23.5 MB 4.8 MB/s eta 0:00:04
   -------------- ------------------------- 8.4/23.5 MB 4.9 MB/s eta 0:00:04
   -------------- ------------------------- 8.7/23.5 MB 4.8 MB/s eta 0:00:04
   ----------------- ---------------------- 10.2/23.5 MB 4.8 MB/s eta 0:00:03
   ------------------ --------------------- 11.0/23.5 MB 4.8 MB/s eta 0:00:03
   -------------------- ------------------- 11.8/23.5 MB 4.7 MB/s eta 0:00:03
   


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# Import chromadb - our vector database
import chromadb

# Create a client - this manages our database connection
# Using an in-memory client for now (data lives only while notebook is running)
chroma_client = chromadb.Client()

# Create a "collection" - think of this like a table in a normal database,
# but specialized for storing vectors
collection = chroma_client.create_collection(name="hr_policy")

# Add our chunks to the collection
# - documents: the actual text of each chunk
# - embeddings: the vector (numbers) we generated for each chunk
# - ids: a unique identifier for each chunk (required by Chroma)
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),  # convert to plain list format Chroma expects
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Added {collection.count()} chunks to the vector store")

Added 10 chunks to the vector store


In [23]:
# The question we want to ask
query = "How many sick leave days do I get?"

# Convert the question into an embedding, using the SAME model we used for the chunks
# (This is important - query and chunks must use the same embedding model to be comparable)
query_embedding = embedding_model.encode([query])

# Search the vector store for the most similar chunks
# n_results = how many top matches to return
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

# Print the most relevant chunk(s) found
print("Question:", query)
print("\n--- Most relevant chunk(s) found ---\n")
for doc in results['documents'][0]:
    print(doc)
    print("\n---\n")

Question: How many sick leave days do I get?

--- Most relevant chunk(s) found ---

required for sick leave exceeding 2 consecutive days. Sick leave may be carried forward up to a
maximum of 15 days into the following year.
4. Earned / Privilege Leave
Employees accrue 18 (eighteen) earned leave days per calendar year, credited at 1.5 days per
month after completion of the probation period. Earned leave may be carried forward, up to a
maximum accumulation of 45 days. Employees may encash up to 10 accumulated earned leave
days per year, subject to manager and HR approval.

---

day per month. Casual leave is intended for short-term personal needs and cannot be availed for
more than 3 consecutive days without prior approval from the reporting manager.
Unused casual leave cannot be carried forward to the next calendar year and will lapse on
December 31st.
3. Sick Leave
Employees are entitled to 10 (ten) sick leave days per calendar year. A medical certificate is
required for sick leave exc

In [24]:
def ask_hr_policy(question):
    """
    Given a question, this function:
    1. Converts the question into an embedding
    2. Searches the vector store for the most relevant chunks
    3. Sends those chunks + the question to the LLM
    4. Returns the LLM's grounded answer
    """
    # Step 1: embed the question
    query_embedding = embedding_model.encode([question])

    # Step 2: retrieve top 3 relevant chunks
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )
    retrieved_chunks = results['documents'][0]

    # Combine the retrieved chunks into one block of context text
    context = "\n\n".join(retrieved_chunks)

    # Step 3: build a prompt that includes the context + the question
    prompt = f"""You are an HR policy assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say "This isn't covered in the provided policy documents."

Context:
{context}

Question: {question}

Answer:"""

    # Step 4: send it to the LLM
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# Test it
answer = ask_hr_policy("How many sick leave days do I get?")
print(answer)

You are entitled to **10 sick leave days per calendar year.**


In [25]:
print(ask_hr_policy("What is the notice period for a manager?"))
print("\n---\n")
print(ask_hr_policy("Can I carry forward my casual leave?"))
print("\n---\n")
print(ask_hr_policy("What is the company's parking policy?"))  # this should trigger the "not covered" fallback

The notice period for a manager is 60 (sixty) days.

---

No. Unused casual leave cannot be carried forward to the next calendar year and will lapse on December 31st.

---

This isn't covered in the provided policy documents.


In [26]:
def plan_action(question):
    """
    Decides whether a question needs document retrieval or can be answered directly.
    Returns either "RETRIEVE" or "DIRECT"
    """
    planning_prompt = f"""You are a planning agent. Decide how to handle this user message.

If the message is a greeting, small talk, or doesn't require looking up HR policy 
information (e.g. "hello", "thank you", "who are you"), respond with exactly: DIRECT

If the message is a genuine question about HR policy, leave, benefits, or company rules 
that requires looking up a document, respond with exactly: RETRIEVE

Message: "{question}"

Respond with ONLY one word: DIRECT or RETRIEVE"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": planning_prompt}]
    )

    decision = response.choices[0].message.content.strip().upper()
    return decision

# Test it
print(plan_action("Hi there!"))
print(plan_action("How many sick leave days do I get?"))

DIRECT
RETRIEVE


In [27]:
def smart_assistant(question):
    """
    The full agent flow:
    1. Planner decides: does this need document retrieval or a direct reply?
    2. If DIRECT -> answer normally without searching documents
    3. If RETRIEVE -> run the full RAG pipeline (search + grounded answer)
    """
    # Step 1: Ask the planner what to do
    decision = plan_action(question)
    
    if decision == "DIRECT":
        # Just have a normal conversation, no document lookup needed
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "You are a friendly HR assistant chatbot."},
                {"role": "user", "content": question}
            ]
        )
        return response.choices[0].message.content
    
    else:  # RETRIEVE
        # Use our existing RAG pipeline
        return ask_hr_policy(question)

# Test with different types of questions
print("Q: Hi there!")
print(smart_assistant("Hi there!"))
print("\n---\n")
print("Q: How many sick leave days do I get?")
print(smart_assistant("How many sick leave days do I get?"))

Q: Hi there!
Hello! 👋 I’m here to help with any HR questions or support you might need. How can I assist you today?

---

Q: How many sick leave days do I get?
10 sick leave days per calendar year.


In [28]:
def validate_answer(question, context, answer):
    """
    Checks whether the answer is actually supported by the retrieved context.
    Returns True if grounded, False if it looks like it might be hallucinated.
    """
    validation_prompt = f"""You are a strict fact-checker. Determine if the ANSWER below 
is fully supported by the CONTEXT. Reply with only YES or NO.

Context:
{context}

Question: {question}
Answer: {answer}

Is the answer fully supported by the context? Reply with only YES or NO."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": validation_prompt}]
    )

    verdict = response.choices[0].message.content.strip().upper()
    return "YES" in verdict

In [29]:
def ask_hr_policy(question):
    """
    Full RAG pipeline with validation:
    1. Embed the question
    2. Retrieve relevant chunks
    3. Generate an answer using those chunks
    4. Validate the answer is actually grounded before returning it
    """
    query_embedding = embedding_model.encode([question])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )
    retrieved_chunks = results['documents'][0]
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""You are an HR policy assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say "This isn't covered in the provided policy documents."

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content

    # Validate the answer before returning it
    is_valid = validate_answer(question, context, answer)
    
    if not is_valid:
        return "I found some related information, but I'm not confident it fully answers your question. Please verify with HR directly."
    
    return answer

# Test it again
print(ask_hr_policy("How many sick leave days do I get?"))

You are entitled to 10 sick leave days per calendar year.


In [30]:
%pip install streamlit

   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.5 MB 6.7 MB/s eta 0:00:02
   ------- -------------------------------- 2.1/10.5 MB 6.2 MB/s eta 0:00:02
   --------------- ------------------------ 4.2/10.5 MB 7.4 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.5 MB 8.0 MB/s eta 0:00:01
   --------------------------------- ------ 8.9/10.5 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.5 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 8.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   ---------------------------------------- 797.6/797.6 kB 5.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------ --------------------------- 3.1/9.8 MB 12.3 MB/s eta 0:00:01
   -------------- ------------------------- 3.7/9.8 MB 10.4 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
%pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
